In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt #for igures
%matplotlib inline

In [5]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8] #check

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
#built wocabulary of characters and mappings to from integers 
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.']= 0
itos={i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [7]:
# buil the dataset
block_size=3 #context lenght :how many charakter do we take to predict next one 
X , Y = [] , []
for w in words[:5]:#firstly try with only 5 word = 32 sample
    print(w)
    contex = [0] * block_size
    for ch in w + '.' :
        ix =stoi[ch]
        X.append(contex)
        Y.append(ix)
        print(''.join(itos[i] for i in contex), '---->',itos[ix])
        contex=contex[1:]+[ix] #crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [10]:
X.shape,X.dtype, Y.shape,Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [8]:
#look up table 
#in the paper mention on the video , in 25000 word each word haves their 30 dimension embeddings
#for our neural network we have 3 charecter to predict nex one from 27 probable character
#so we should do embed them into small number of dimension like 2 
#as well as our network train the characters probabilities will move in the 2 dimention space and get far and naer with each other
#lets look up table for each 27 charecter for deir embeddings

C= torch.randn(27,2)

In [9]:
C[5]

tensor([ 0.1113, -0.6319])

In [10]:
# We can represent each sample using F.one_hot.
# one_hot produces a vector of length 27 for each index:
# every position is 0, except the position at that index, which is 1.
F.one_hot(torch.tensor(5),num_classes=27  )


tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [ ]:
#we had to casting float to multipy a float with int
#matrix dimensions will be [1,27][27,2] = [1,2]
F.one_hot(torch.tensor(5),num_classes=27  ).float() @ C

tensor([ 0.1113, -0.6319])

In [12]:
#we will contunie with just index them
#C[[5,6,7]]
C[torch.tensor([5,6,7,7,6,6,6,5])]
#C[X].shape


tensor([[ 0.1113, -0.6319],
        [ 0.2105,  0.7535],
        [-2.4850,  1.4897],
        [-2.4850,  1.4897],
        [ 0.2105,  0.7535],
        [ 0.2105,  0.7535],
        [ 0.2105,  0.7535],
        [ 0.1113, -0.6319]])

In [ ]:
# every character index in X is replaced by its 2-dimensional vector (row) from C
emb=C[X] #C[X][2] created = C [32,3,2] 
emb.shape

torch.Size([32, 3, 2])

In [18]:
#lets add a hidden layer
w1= torch.randn((6,100)) #100neuron is optional
b1=torch.randn(100)

In [14]:
# emb @ w1 + b1
emb[:, 0, :]##each 32 target samples first embed in 3 previous



tensor([[ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1135e-01, -6.3192e-01],
        [ 1.9680e+00,  4.2262e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [-1.0872e-01, -1.0687e+00],
        [ 6.9818e-01, -5.8945e-01],
        [-3.3036e-01,  4.3679e-01],
        [ 4.4107e-01,  1.3143e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 3.5481e-01, -2.5670e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [-3.3036e-01,  4.3679e-01],
        [-1.0879e+00, -4.8532e-01],
        [ 3.5481e-01, -2.5670e-01],
        [-3.4008e-01, -1.3612e-01],
        [ 1.1135e-01, -6.3192e-01],
        [ 6.9818e-01, -5.8945e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.7158e-01],
        [ 1.1199e+00, -2.715

In [99]:
torch.cat([emb[:,0,:], emb[:,1,:],   emb[:,2,:]], 1).shape 
#each row represent one targets  [x1,x2,x3,y1,y2,y3] embeds, x and y is dimentions 123 is previous points embeds


torch.Size([228146, 6])

In [54]:

torch.cat(torch.unbind(emb,1),1)
#each row represent one targets  [x1,x2,x3,y1,y2,y3] embeds, x and y is dimentions 123 is previous points embeds
#this way creates a hole new memory so this way is not efficient

tensor([[ 0.8573, -0.1472,  0.8573, -0.1472,  0.8573, -0.1472],
        [ 0.8573, -0.1472,  0.8573, -0.1472,  0.3934,  0.1480],
        [ 0.8573, -0.1472,  0.3934,  0.1480, -0.1334, -0.0599],
        ...,
        [-0.0747, -0.1085, -0.0747, -0.1085,  0.2605, -0.1197],
        [-0.0747, -0.1085,  0.2605, -0.1197, -0.0747, -0.1085],
        [ 0.2605, -0.1197, -0.0747, -0.1085,  0.0441,  0.0197]],
       grad_fn=<CatBackward0>)

In [ ]:
emb.shape[0]# number of data samples

32

In [ ]:
emb.view(emb.shape[0],6)#this is more efficient to pytorch tensor structure 


#emb.view(-1,6) will also work   # ? × 6 = 192 -->32
#.view extremly efficient

tensor([[ 1.6898,  0.2825,  1.6898,  0.2825,  1.6898,  0.2825],
        [ 1.6898,  0.2825,  1.6898,  0.2825, -0.3098, -0.6113],
        [ 1.6898,  0.2825, -0.3098, -0.6113,  0.7924, -1.2371],
        ...,
        [ 1.9137,  0.1698,  1.9137,  0.1698,  0.0917, -0.9000],
        [ 1.9137,  0.1698,  0.0917, -0.9000,  1.9137,  0.1698],
        [ 0.0917, -0.9000,  1.9137,  0.1698, -0.4685,  1.2703]])

In [16]:
emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)
#in this way we get the exact same resul you can check with run above code line


tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [ ]:
h = torch.tanh(emb.view(-1,6) @ w1 +b1) #this is hidden layer with ativation
h.shape #32 train data -> 32 hidden layer consist of 100 neurons

torch.Size([32, 100])

In [21]:
w2= torch.randn((100,27))
b2= torch.randn(27)

In [22]:
logits= h @ w2 + b2
logits.shape# final layer

torch.Size([32, 27])

In [28]:
counts= logits.exp() #for negatives 

In [29]:
prob = counts / counts.sum(1, keepdims= True)# for each input sample there is 27 charecters probabilities for next come 

prob.shape
#now thats gives us the a row of 27 probibilites for each sample.
#max prob will be our max likely prediction

torch.Size([32, 27])

In [30]:
prob[0].sum()

tensor(1.)

In [26]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [ ]:
#this cell work when we select first 32 sample
prob[torch.arange(32), Y] #only correct chars probability for each sample
# each row and Y in index of probability of correct characters

loss = -prob[torch.arange(32), Y].log().mean()
#we need log and mean to find loss
loss
#al these are not efficient we will use cross enthrophy but main idea is same

tensor(17.4441)

In [32]:
# for exact same result we can :
loss =F.cross_entropy(logits, Y)
loss



tensor(17.4441)

In [43]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
w1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
w2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, b1, w2, b2]

In [44]:
sum(p.nelement() for p in parameters)#number of all parameters in the network

3481

In [48]:
parameters = [C, w1, b1, w2, b2]
for p in parameters:
    p.requires_grad = True

In [71]:

for _ in range (100):
    #Forwardpass
    emb=C[X] #[32,3,2]
    h=torch.tanh(emb.view(-1,6)@ w1 + b1)
    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, Y)
    print (loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1 *p.grad


0.2541852593421936
0.2541869282722473
0.2541816234588623
0.2541831433773041
0.25417789816856384
0.2541794776916504
0.2541741728782654
0.25417572259902954
0.2541704475879669
0.25417202711105347
0.25416675209999084
0.254168301820755
0.25416311621665955
0.2541646361351013
0.2541593909263611
0.25416091084480286
0.2541557252407074
0.25415724515914917
0.2541520595550537
0.2541535794734955
0.2541484534740448
0.2541499733924866
0.2541448175907135
0.2541463375091553
0.2541411519050598
0.254142701625824
0.2541375458240509
0.2541390657424927
0.254133939743042
0.2541354298591614
0.25413039326667786
0.25413185358047485
0.2541268467903137
0.2541283071041107
0.2541232407093048
0.2541247010231018
0.2541196942329407
0.25412115454673767
0.25411608815193176
0.25411754846572876
0.2541125416755676
0.2541140019893646
0.2541090250015259
0.2541104555130005
0.25410544872283936
0.25410693883895874
0.25410196185112
0.2541033923625946
0.25409844517707825
0.25409990549087524
0.2540949583053589
0.2540963590145111
0

In [73]:
logits.max(1)#for every dimention 

torch.return_types.max(
values=tensor([13.7592, 18.6681, 20.9845, 21.1941, 17.3517, 13.7592, 16.5834, 14.6828,
        16.4185, 19.0863, 16.6159, 21.5653, 13.7592, 17.8675, 17.8242, 20.7804,
        13.7592, 17.1329, 15.9207, 17.7885, 19.1312, 16.6707, 11.5014, 11.2358,
        15.9393, 13.7592, 16.7685, 17.5433, 13.1721, 16.6847, 19.8099, 16.8845],
       grad_fn=<MaxBackward0>),
indices=tensor([ 9, 13, 13,  1,  0,  9, 12,  9, 22,  9,  1,  0,  9, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0,  9, 15, 16,  8,  9,  1,  0]))

In [74]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [66]:
torch.randint(0, X.shape[0], (32,))

tensor([ 1, 23, 25, 18,  7, 28,  7, 16, 22, 14,  0, 10,  6,  2, 20, 21,  4,  5,
         6, 31, 12,  5, 28, 11, 10,  5, 14, 20, 28,  7, 10, 30])